# Trigger Arc TSP: ekzekutimi dhe analiza (SA, GRASP, ILS)

Ky notebook:
1. e merr projektin (nga Dataset-i i Kaggle-it ose nga GitHub-i);
2. ekzekuton `src/main.py` me të tria algoritmet: **Simulated Annealing**, **GRASP** dhe **Iterated Local Search**;
3. i analizon rezultatet: tabela, krahasimi dhe grafikë;
4. i paketon rezultatet në `rezultatet.zip` për shkarkim.

**Në Kaggle:**
- *Settings → Accelerator: None (CPU)*. GPU-ja nuk ndihmon, sepse algoritmet janë sekuenciale.
- Ose shtoje repo-n si **Dataset** (*Add Input*), ose ndize *Settings → Internet: On* që të klonohet nga GitHub-i.
- `grf15.txt` dhe `grf16.txt` nuk janë në GitHub. Nëse i do, ngarkoji në një Dataset dhe shtoje atë me *Add Input*: notebook-u i gjen vetë.
- Për ekzekutime të gjata përdor **Save Version → Save & Run All**. Kështu notebook-u punon në sfond edhe kur e mbyll shfletuesin.

## 1. Parametrat
Ndryshoji këtu. Të gjitha algoritmet marrin kohë të barabartë për çdo provë.

In [ ]:
# Cilat instanca: "te_gjitha" ose listë, p.sh. ["grf7", "grf9"]
INSTANCAT = ["grf1", "grf2", "grf3", "grf4", "grf5", "grf6", "grf7", "grf8", "grf9",
             "grf10", "grf11", "grf12", "grf13", "grf14", "grf15", "grf16", "grf17", "grf18"]
# (grf19, grf20, grf21 janë kopje të grf2, grf8, grf14, prandaj nuk janë në listë)

ALGORITMET = ["sa", "grasp", "ils"]
SEEDS = [1, 2, 3, 4, 5]
KOHA = 60          # sekonda për çdo provë
PROCESE = 4        # Kaggle CPU ka 4 bërthama

EKZEKUTO = True    # False = vetëm analizo rezultatet ekzistuese, pa ekzekutuar
FILLO_NGA_ZERO = True   # True = fshin rezultatet e vjetra të SA/GRASP/ILS (vetëm në kopjen e Kaggle-it),
                        # që të mos përzihen me provat e reja (p.sh. grf1 i ekzekutuar në PyCharm)

GITHUB_URL = "https://github.com/Pranvera08/Pranvera-Gashi-The-Trigger-Arc-TSP.git"

## 2. Përgatitja e projektit

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

KAGGLE = Path("/kaggle/input").exists()

def gjej_projektin_lokal():
    """Kur notebook-u hapet lokalisht nga folderi notebooks/ i projektit."""
    for d in [Path.cwd(), Path.cwd().parent]:
        if (d / "src" / "main.py").exists():
            return d
    return None

if KAGGLE:
    PROJEKTI = Path("/kaggle/working/projekti")
    if not PROJEKTI.exists():
        burimi = next(Path("/kaggle/input").rglob("src/main.py"), None)
        if burimi is not None:
            # /kaggle/input është vetëm për lexim, prandaj kopjohet te /kaggle/working
            shutil.copytree(burimi.parent.parent, PROJEKTI)
            print("Projekti u kopjua nga Dataset-i:", burimi.parent.parent)
        else:
            r = subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(PROJEKTI)],
                               capture_output=True, text=True)
            if r.returncode != 0:
                raise RuntimeError("Klonimi dështoi. Ndize Settings → Internet: On, "
                                   "ose shtoje projektin si Dataset.\n" + r.stderr)
            print("Projekti u klonua nga GitHub-i.")
    # Instancat shtesë (p.sh. grf15, grf16) nga çdo Dataset i shtuar
    for f in Path("/kaggle/input").rglob("grf*.txt"):
        cak = PROJEKTI / "instances" / f.name
        if "instances" in f.parts and not cak.exists():
            shutil.copy(f, cak)
            print("U shtua instanca:", f.name)
else:
    PROJEKTI = gjej_projektin_lokal()
    if PROJEKTI is None:
        raise RuntimeError("Hape notebook-un nga folderi i projektit (notebooks/).")

REZULTATET = PROJEKTI / "results"

if KAGGLE and FILLO_NGA_ZERO and EKZEKUTO:
    for alg in ["sa", "grasp", "ils"]:
        shutil.rmtree(REZULTATET / alg, ignore_errors=True)
    (REZULTATET / "krahasimi.csv").unlink(missing_ok=True)
    print("Rezultatet e vjetra të SA/GRASP/ILS u fshinë (results/sa_v1 mbetet).")

print("Projekti:", PROJEKTI)
print("Instancat e disponueshme:", sorted(p.stem for p in (PROJEKTI / "instances").glob("*.txt")))

## 3. Ekzekutimi i algoritmeve
Rezultatet shfaqen pas çdo instance. Ato që ekzistojnë nuk fshihen: provat e reja u shtohen CSV-ve.

In [ ]:
import time

ekzistojne = {p.stem for p in (PROJEKTI / "instances").glob("grf*.txt")}
lista = sorted(ekzistojne, key=lambda s: int(s[3:])) if INSTANCAT == "te_gjitha" else INSTANCAT
mungojne = [i for i in lista if i not in ekzistojne]
if mungojne:
    print("Kujdes, këto instanca mungojnë dhe kapërcehen:", mungojne)
lista = [i for i in lista if i in ekzistojne]

provat = len(lista) * len(ALGORITMET) * len(SEEDS)
print(f"{len(lista)} instanca × {len(ALGORITMET)} algoritme × {len(SEEDS)} seed-e = {provat} prova")
print(f"Koha e përafërt: {provat * KOHA / PROCESE / 60:.0f} minuta (pa kohën e leximit)")

if EKZEKUTO and lista:
    komanda = [sys.executable, "-u", "src/main.py", *lista,
               "--alg", *ALGORITMET, "--seeds", *map(str, SEEDS),
               "--koha", str(KOHA), "--procese", str(PROCESE)]
    fillimi = time.time()
    p = subprocess.Popen(komanda, cwd=PROJEKTI, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for rreshti in p.stdout:
        print(rreshti, end="")
    p.wait()
    print(f"\nPërfundoi për {(time.time() - fillimi) / 60:.1f} minuta (kodi {p.returncode}).")
    if p.returncode != 0:
        raise RuntimeError("main.py përfundoi me gabim, shih daljen më lart.")
else:
    print("Ekzekutimi u kapërcye (EKZEKUTO = False).")

## 4. Leximi i rezultateve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

EMRAT = {"sa": "SA", "grasp": "GRASP", "ils": "ILS"}
# Ngjyrat: tre slotet e para të një palete të kontrolluar për daltonizëm
NGJYRAT = {"sa": "#2a78d6", "grasp": "#eb6834", "ils": "#1baf7a"}

def rendit(s):
    return s.map(lambda x: int(x[3:]) if x.startswith("grf") and x[3:].isdigit() else 999)

pjeset = []
for alg in ALGORITMET:
    f = REZULTATET / alg / "provat.csv"
    if f.exists() and f.stat().st_size > 0:
        d = pd.read_csv(f)
        d["alg"] = alg
        pjeset.append(d)
provat = pd.concat(pjeset, ignore_index=True)
# vetëm provat me kohën e zgjedhur, që krahasimi të jetë i drejtë
provat = provat[provat["koha_max"] == float(KOHA)]
provat = provat[provat["instanca"].str.startswith("grf")]
if provat.empty:
    raise RuntimeError(f"Nuk ka prova me koha_max = {KOHA}. Ndrysho KOHA ose ekzekuto algoritmet.")

# Kostoja më e mirë e njohur për çdo instancë (nga të gjitha algoritmet)
me_e_mira = provat.groupby("instanca")["kosto"].min().rename("me_e_mira_e_njohur")
provat = provat.join(me_e_mira, on="instanca")
provat["gap_%"] = (provat["kosto"] - provat["me_e_mira_e_njohur"]) / provat["me_e_mira_e_njohur"] * 100

print(f"{len(provat)} prova, {provat['instanca'].nunique()} instanca, koha {KOHA} s për provë")

## 5. Tabela e krahasimit
Për çdo instancë: kostoja më e mirë dhe mesatarja e secilit algoritëm (nga të gjitha seed-et).

In [ ]:
t = provat.groupby(["instanca", "alg"])["kosto"].agg(["min", "mean", "std"]).unstack("alg")
tabela = pd.DataFrame(index=t.index)
for alg in ALGORITMET:
    if ("min", alg) in t.columns:
        tabela[f"{EMRAT[alg]} më e mira"] = t[("min", alg)].round(2)
        tabela[f"{EMRAT[alg]} mesatarja"] = t[("mean", alg)].round(2)
        tabela[f"{EMRAT[alg]} devijimi"] = t[("std", alg)].round(2)
tabela = tabela.loc[sorted(tabela.index, key=lambda x: int(x[3:]))]

def fituesi(rr, kolona):
    vlerat = {a: rr[f"{EMRAT[a]} {kolona}"] for a in ALGORITMET if f"{EMRAT[a]} {kolona}" in rr}
    m = min(vlerat.values())
    fit = [EMRAT[a] for a, v in vlerat.items() if v <= m + 1e-9]
    return fit[0] if len(fit) == 1 else "barazim"

tabela["Fituesi (më e mira)"] = tabela.apply(lambda r: fituesi(r, "më e mira"), axis=1)
tabela["Fituesi (mesatarja)"] = tabela.apply(lambda r: fituesi(r, "mesatarja"), axis=1)
display(tabela)

print("\nSa herë fiton secili algoritëm (sipas mesatares):")
display(tabela["Fituesi (mesatarja)"].value_counts().rename("instanca"))

## 6. Sa larg nga zgjidhja më e mirë (gap)
Kostot e instancave kanë shkallë të ndryshme, prandaj krahasohet **gap-i**: sa përqind është kostoja mbi zgjidhjen më të mirë të njohur për atë instancë. 0% = zgjidhja më e mirë.

In [ ]:
gap = provat.groupby(["instanca", "alg"])["gap_%"].mean().unstack("alg")
gap = gap.loc[sorted(gap.index, key=lambda x: int(x[3:]))]

fig, ax = plt.subplots(figsize=(13, 4.5))
gjeresia = 0.8 / len(ALGORITMET)
x = range(len(gap))
for k, alg in enumerate(ALGORITMET):
    if alg in gap:
        ax.bar([i + (k - (len(ALGORITMET) - 1) / 2) * gjeresia for i in x], gap[alg],
               width=gjeresia * 0.92, color=NGJYRAT[alg], label=EMRAT[alg])
ax.set_xticks(list(x))
ax.set_xticklabels(gap.index, rotation=45)
ax.set_ylabel("Gap mesatar (%)")
ax.set_title(f"Gap-i mesatar ndaj zgjidhjes më të mirë të njohur ({len(SEEDS)} seed-e, {KOHA} s)")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Gap-i mesatar mbi të gjitha instancat (%):")
display(gap.mean().rename(index=EMRAT).round(2))

## 7. Sa luhatet secili algoritëm (boxplot sipas seed-eve)
Kutia tregon shpërndarjen e gap-it të 5 seed-eve për të gjitha instancat bashkë. Kuti më e ngushtë dhe më afër 0 do të thotë algoritëm më i qëndrueshëm.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
te_dhenat = [provat.loc[provat["alg"] == a, "gap_%"] for a in ALGORITMET]
bp = ax.boxplot(te_dhenat, patch_artist=True, widths=0.5)
ax.set_xticks(range(1, len(ALGORITMET) + 1))
ax.set_xticklabels([EMRAT[a] for a in ALGORITMET])
for kuti, alg in zip(bp["boxes"], ALGORITMET):
    kuti.set_facecolor(NGJYRAT[alg]); kuti.set_alpha(0.75)
for m in bp["medians"]:
    m.set_color("#222222")
ax.set_ylabel("Gap (%)")
ax.set_title("Shpërndarja e gap-it për të gjitha provat")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Konvergjenca
Si ulet kostoja më e mirë me kalimin e kohës për një instancë, mesatarisht mbi seed-et. Ndrysho `INSTANCA_KONV` për instancë tjetër.

Kurba e GRASP-it dhe e ILS-it fillon më vonë: në instancat e mëdha kërkimi i parë lokal zgjat disa sekonda para se të ketë zgjidhjen e parë. SA ka zgjidhje që nga sekonda 0, sepse nis nga rruga e backtracking-ut.

In [ ]:
import numpy as np

INSTANCA_KONV = "grf13" if "grf13" in set(provat["instanca"]) else provat["instanca"].iloc[0]

fig, ax = plt.subplots(figsize=(9, 4.5))
pikat = np.linspace(0, KOHA, 200)
for alg in ALGORITMET:
    f = REZULTATET / alg / "konvergjenca.csv"
    if not f.exists() or f.stat().st_size == 0:
        continue
    k = pd.read_csv(f)
    k = k[(k["instanca"] == INSTANCA_KONV) & (k["koha_max"] == float(KOHA))]
    if k.empty:
        continue
    kurbat = []
    for seed, g in k.groupby("seed"):
        g = g.sort_values("sekonda")
        # kostoja më e mirë deri në çastin t (shkallë)
        idx = np.searchsorted(g["sekonda"].values, pikat, side="right") - 1
        v = np.where(idx >= 0, g["kosto"].values[np.clip(idx, 0, None)], np.nan)
        kurbat.append(v)
    m = np.vstack(kurbat)
    # mesatarja vetëm aty ku të gjitha seed-et kanë vlerë (përndryshe kurba "rritet"
    # kur një seed i ri hyn në mesatare me kosto më të lartë)
    mesatarja = np.where(np.isnan(m).any(axis=0), np.nan, m.mean(axis=0))
    ax.plot(pikat, mesatarja, color=NGJYRAT[alg], linewidth=2, label=EMRAT[alg])
ax.set_xlabel("Koha (s)")
ax.set_ylabel("Kostoja më e mirë (mesatare e seed-eve)")
ax.set_title(f"Konvergjenca: {INSTANCA_KONV}")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 9. Krahasimi me versionin e parë (SA v1)
Kostoja më e mirë e versionit të parë (vetëm SA, 500 000 iteracione / 60 s) kundrejt më të mirës së algoritmeve të reja.

In [ ]:
v1_f = REZULTATET / "sa_v1" / "best_results.csv"
if v1_f.exists():
    v1 = pd.read_csv(v1_f).set_index("instance")["best_cost"].rename("SA v1")
    reja = provat.groupby("instanca")["kosto"].min().rename("Më e mira e re")
    kr = pd.concat([v1, reja], axis=1, join="inner")
    kr["Përmirësimi (%)"] = ((kr["SA v1"] - kr["Më e mira e re"]) / kr["SA v1"] * 100).round(2)
    kr = kr.loc[sorted(kr.index, key=lambda x: int(x[3:]))]
    display(kr)
else:
    print("Nuk u gjet results/sa_v1/best_results.csv")

## 10. Shkarkimi i rezultateve
Krijon `rezultatet.zip` me gjithë folderin `results/`. Në Kaggle e gjen te *Output* (paneli djathtas, ose skeda Output e versionit të ruajtur).

In [ ]:
if KAGGLE:
    zip_f = shutil.make_archive("/kaggle/working/rezultatet", "zip", REZULTATET)
    print("U krijua:", zip_f)
    print("Pas shkarkimit, nxirre përmbajtjen te folderi results/ i projektit dhe bëj commit.")
else:
    print("Lokalisht rezultatet janë tashmë te:", REZULTATET)